<a href="https://colab.research.google.com/github/zebas-hidalgo/dia_del_libro2026/blob/main/bloques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# BLOQUE 1: ENTORNO Y CEREBRO IA (V8.1)
# ==========================================
import os
import subprocess
import time
import locale
import requests

# Evita errores de codificación en Colab
locale.getpreferredencoding = lambda: "UTF-8"

print("1. Instalando dependencias del sistema (zstd, ffmpeg)... wheel")
!apt-get update -qq && apt-get install -y -qq ffmpeg zstd

print("\n2. Instalando librerías de Python...")
!pip install -q openai-whisper gradio edge-tts moviepy==1.0.3 numpy pillow

print("\n3. Instalando motor local (Ollama)...")
!curl -fsSL https://ollama.com/install.sh | sh

print("\n4. Iniciando servidor Ollama...")
subprocess.Popen(["nohup", "ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Verificación de que el servidor está activo
for i in range(10):
    try:
        requests.get("http://localhost:11434/", timeout=1)
        print("✅ Servidor Ollama detectado y activo.")
        break
    except:
        if i == 9:
            print("❌ No se pudo conectar con Ollama tras 10 intentos.")
        else:
            time.sleep(2)

print("\n5. Descargando modelo de IA (Phi-3)... Esto puede tardar un poco.")
!ollama pull phi3

print("\n✅ Bloque 1 Listo. El cerebro está preparado. Ahora puedes ejecutar los siguientes bloques.")

In [ ]:
# ==========================================
# DIAGNÓSTICO DE OLLAMA
# ==========================================
import requests
import subprocess

print("--- Verificando Servidor ---")
try:
    res = requests.get('http://localhost:11434/api/tags')
    if res.status_code == 200:
        print("✅ El servidor Ollama está RESPONDIENDO.")
        print("Modelos disponibles:", res.json())
    else:
        print(f"⚠️ El servidor respondió con código: {res.status_code}")
except Exception as e:
    print(f"❌ El servidor NO responde: {e}")

print("\n--- Verificando Procesos ---")
!ps aux | grep ollama

In [ ]:
# ==========================================
# BLOQUE 2: LÓGICA DE ANIMACIÓN (V8.0)
# ==========================================
import requests
import os
import time
import subprocess
import whisper
import edge_tts
from PIL import Image
import numpy as np
from moviepy.editor import ImageSequenceClip, AudioFileClip

print("Cargando los oídos de la IA (Whisper Tiny)...")
whisper_model = whisper.load_model("tiny")

def asegurar_ollama():
    try:
        requests.get("http://localhost:11434/", timeout=2)
    except:
        subprocess.Popen(["nohup", "ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(5)

def crear_video_dinamico(audio_path, ruta_cerrada, ruta_abierta):
    # Salvavidas: Si el estudiante olvidó subir una foto en la web, creamos cuadros de colores
    if not ruta_cerrada or not os.path.exists(str(ruta_cerrada)):
        ruta_cerrada = "/content/default_c.jpg"
        Image.new("RGB", (400, 400), "#2C3E50").save(ruta_cerrada) # Azul oscuro
    if not ruta_abierta or not os.path.exists(str(ruta_abierta)):
        ruta_abierta = "/content/default_a.jpg"
        Image.new("RGB", (400, 400), "#E74C3C").save(ruta_abierta) # Rojo

    # Limpieza de imágenes
    img_c = Image.open(ruta_cerrada).convert("RGB")
    img_a = Image.open(ruta_abierta).convert("RGB")

    # Forzar dimensiones pares para que funcione en Chrome/Safari
    w, h = img_c.size
    w, h = w - (w % 2), h - (h % 2)
    img_c, img_a = img_c.resize((w, h)), img_a.resize((w, h))

    arr_c, arr_a = np.array(img_c), np.array(img_a)

    try:
        audio = AudioFileClip(audio_path)
    except:
        return None

    # Ajustamos a 6 cuadros por segundo para un habla natural
    fps = 6
    total_frames = max(int(audio.duration * fps), fps)
    frames = [arr_c if i % 2 == 0 else arr_a for i in range(total_frames)]
    frames.append(arr_c) # Siempre termina con la boca cerrada

    # Renderizamos el video
    clip = ImageSequenceClip(frames, fps=fps).set_audio(audio)
    salida = f"/content/out_{int(time.time())}.mp4"
    clip.write_videofile(salida, fps=fps, codec="libx264", audio_codec="aac", logger=None)

    clip.close()
    audio.close()
    return salida

async def procesar_studio(audio_mic, texto, prompt, voz_elegida, img_cerrada, img_abierta, hist_gradio, hist_ollama):
    asegurar_ollama()

    # 1. Escuchar al usuario
    if texto and texto.strip():
        user_msg = texto
    elif audio_mic:
        user_msg = whisper_model.transcribe(audio_mic, language="es")["text"]
    else:
        return None, hist_gradio, hist_ollama, "⚠️ Escribe o di algo primero."

    # 2. Gestionar la memoria (Solo recuerda los últimos 2 intercambios para no saturarse)
    hist_ollama = [hist_ollama[0]] + hist_ollama[-4:] if len(hist_ollama) > 5 else hist_ollama
    hist_ollama.append({"role": "user", "content": user_msg})

    # Inyectamos la personalidad que el estudiante definió
    hist_ollama[0] = {"role": "system", "content": prompt}

    # 3. Pensar la respuesta
    try:
        res = requests.post("http://localhost:11434/api/chat", json={"model": "phi3", "messages": hist_ollama, "stream": False}).json()["message"]["content"]
    except:
        hist_ollama.pop()
        return None, hist_gradio, hist_ollama, "❌ Error de conexión con la IA"

    hist_ollama.append({"role": "assistant", "content": res})

    # 4. Generar la Voz (Usando el acento seleccionado en la web)
    texto_limpio = res.split("]")[1].strip() if "]" in res else res
    audio_tmp = f"/content/voz_{int(time.time())}.mp3"
    await edge_tts.Communicate(texto_limpio, voz_elegida).save(audio_tmp)

    # 5. Generar la Animación Visual
    vid = crear_video_dinamico(audio_tmp, img_cerrada, img_abierta)
    hist_gradio.append((user_msg, texto_limpio))

    return vid, hist_gradio, hist_ollama, "✅ Avatar Animado con Éxito"

In [ ]:
# ==========================================
# BLOQUE 3: INTERFAZ WEB (V8.0 STUDIO)
# ==========================================
import gradio as gr

PROMPT_SISTEMA_DEFAULT = "Eres un científico loco. Responde de forma muy breve (máximo 15 palabras) usando términos científicos."

with gr.Blocks(theme=gr.themes.Base()) as demo:
    gr.Markdown("# 🛠️ Avatar Studio: Inteligencia Artificial en el Aula")
    gr.Markdown("**Paso 1:** Construye tu Avatar a la izquierda. **Paso 2:** Interactúa con él a la derecha.")

    estado_historial_gradio = gr.State([])
    estado_historial_ollama = gr.State([{"role": "system", "content": PROMPT_SISTEMA_DEFAULT}])

    with gr.Row():
        # COLUMNA IZQUIERDA: Herramientas de Construcción
        with gr.Column(scale=1):
            with gr.Accordion("⚙️ Configuración del Personaje", open=True):
                in_prompt_sistema = gr.Textbox(
                    value=PROMPT_SISTEMA_DEFAULT,
                    label="🧠 Define la Personalidad",
                    lines=3
                )

                # Lista de voces gratuitas de Microsoft
                in_voz = gr.Dropdown(
                    choices=[
                        ("🇨🇱 Mujer Chilena (Catalina)", "es-CL-CatalinaNeural"),
                        ("🇨🇱 Hombre Chileno (Lorenzo)", "es-CL-LorenzoNeural"),
                        ("🇲🇽 Mujer Mexicana (Dalia)", "es-MX-DaliaNeural"),
                        ("🇪🇸 Hombre Español (Alvaro)", "es-ES-AlvaroNeural"),
                        ("🇦🇷 Hombre Argentino (Tomas)", "es-AR-TomasNeural")
                    ],
                    value="es-CL-CatalinaNeural",
                    label="🗣️ Selecciona la Voz del Personaje"
                )

                gr.Markdown("📸 **Arrastra las imágenes de tu personaje aquí:**")
                with gr.Row():
                    in_img_cerrada = gr.Image(type="filepath", label="Boca Cerrada")
                    in_img_abierta = gr.Image(type="filepath", label="Boca Abierta")

            out_video = gr.Video(label="Pantalla del Avatar", autoplay=True)
            out_estado = gr.Markdown("**Estado:** Esperando configuración...")
            btn_limpiar = gr.Button("🗑️ Borrar Memoria de Conversación")

        # COLUMNA DERECHA: El Chat
        with gr.Column(scale=1):
            pantalla_chat = gr.Chatbot(label="Historial de Chat", height=450)

            with gr.Row():
                in_audio = gr.Audio(sources=["microphone"], type="filepath", label="🎙️ Háblale por micrófono")
            with gr.Row():
                in_text = gr.Textbox(label="⌨️ O escríbele aquí", placeholder="Inicia la conversación...")

            btn_enviar = gr.Button("🎬 Enviar Mensaje", variant="primary")

    # Lógica de los botones
    btn_enviar.click(
        fn=procesar_studio,
        inputs=[
            in_audio, in_text, in_prompt_sistema, in_voz,
            in_img_cerrada, in_img_abierta,
            estado_historial_gradio, estado_historial_ollama
        ],
        outputs=[out_video, pantalla_chat, estado_historial_ollama, out_estado]
    ).then(
        fn=lambda: (None, ""), inputs=None, outputs=[in_audio, in_text]
    )

    def resetear_memoria(prompt_actual):
        return None, [], [{"role": "system", "content": prompt_actual}], "**Estado:** Memoria reiniciada."

    btn_limpiar.click(
        fn=resetear_memoria,
        inputs=[in_prompt_sistema],
        outputs=[out_video, pantalla_chat, estado_historial_ollama, out_estado]
    )

demo.queue()
demo.launch(share=True, debug=True)